# 04 · Para onde foram os votos de Lula (hipóteses A6 e A8)

**Testes confirmatórios.** Especificações registradas em `docs/hipoteses.md` (A6 e A8)
antes do cálculo.

Lula teve 57,1 milhões de votos no 1º turno de 2022 e 53,7 milhões no de 2026 (sem o
exterior). A pergunta é **de onde** veio essa diferença:

- **A6:** de eleitores que deixaram de votar (desmobilização) ou de mudança de voto?
- **A8:** os municípios pequenos pesaram na perda mais do que o seu tamanho?

**Decomposição.** Os votos de Lula num município são o produto de quatro fatores:

| fator | o que mede |
|---|---|
| eleitorado | eleitores aptos |
| comparecimento | quem votou / aptos |
| votos válidos | válidos / quem votou (o resto é branco ou nulo) |
| fatia de Lula | votos de Lula / válidos |

A variação dos votos se divide exatamente entre os quatro (método LMDI, que soma sem resíduo
e permite somar municípios). Exemplo: se o eleitorado cresce 10% e o resto fica igual, toda
a variação vai para "eleitorado".

In [ ]:
import warnings

import altair as alt
import polars as pl
import pyfixest as pf

from kingsrace import paths
from kingsrace.features.decomposicao import decompoe_votos
from kingsrace.viz import tema  # noqa: F401  (registra e ativa o tema do projeto)
from kingsrace.viz.tema import DIVERGENTE, SERIES, TINTA

alt.renderers.enable("jupyter", offline=True)
warnings.filterwarnings("ignore", message=".*singleton fixed effect")

P = paths.PROCESSED
amc = pl.read_parquet(P / "painel_amc.parquet").filter(pl.col("turno") == 1)
municipios = pl.read_parquet(P / "municipios.parquet")
geo = municipios.group_by("amc_id").agg(pl.col("regiao", "cd_regiao_imediata").first())


def pleito(ano, sufixo):
    return amc.filter(pl.col("ano") == ano).select(
        "amc_id",
        *(pl.col(c).alias(f"{c}_{sufixo}") for c in ("aptos", "comparecimento", "validos")),
        pl.col("votos_pt").alias(f"votos_{sufixo}"),
        pl.col("pt_2p_logit").alias(f"pt_logit_{sufixo}"),
    )


base = decompoe_votos(pleito(2022, "de").join(pleito(2026, "para"), on="amc_id")).join(
    geo, on="amc_id"
)

## A6 · Desmobilização ou mudança de voto?

**Critério registrado:** A6 se sustenta se comparecimento + votos válidos responderem por
pelo menos 25% da perda nacional.

In [ ]:
nomes = {
    "c_eleitorado": "Eleitorado",
    "c_comparecimento": "Comparecimento",
    "c_validos": "Brancos e nulos",
    "c_fatia": "Fatia de Lula",
}
total = base.select(pl.col("votos_de", "votos_para", "delta", *nomes).sum())
perda = total["delta"][0]
pl.DataFrame(
    [
        {
            "componente": rotulo,
            "milhões de votos": round(total[c][0] / 1e6, 3),
            "% da variação": round(100 * total[c][0] / perda, 1),
        }
        for c, rotulo in nomes.items()
    ]
)

In [ ]:
desmob = (total["c_comparecimento"][0] + total["c_validos"][0]) / perda
print(f"Variação total: {perda / 1e6:+.2f} milhões de votos")
print(f"Comparecimento + válidos: {100 * desmob:.1f}% da perda (critério: ≥ 25%)")

**Como ler a cascata:** começa nos votos de Lula em 2022 e soma cada componente até chegar
em 2026. Barras vermelhas somam votos a Lula; azuis tiram (convenção do projeto: vermelho =
rumo ao PT, azul = rumo ao adversário).

In [ ]:
passos = [("2022", total["votos_de"][0], "total")]
passos += [(rotulo, total[c][0], "componente") for c, rotulo in nomes.items()]
linhas, acumulado = [], 0.0
for rotulo, valor, tipo in passos:
    inicio = 0.0 if tipo == "total" else acumulado
    fim = valor if tipo == "total" else acumulado + valor
    acumulado = fim
    sinal = "total" if tipo == "total" else ("ganho" if valor > 0 else "perda")
    linhas.append(
        {
            "passo": rotulo,
            "inicio": inicio / 1e6,
            "fim": fim / 1e6,
            "sinal": sinal,
            "valor": valor / 1e6,
        }
    )
linhas.append(
    {
        "passo": "2026",
        "inicio": 0.0,
        "fim": acumulado / 1e6,
        "sinal": "total",
        "valor": acumulado / 1e6,
    }
)
cascata = pl.DataFrame(linhas)
ordem = cascata["passo"].to_list()
cores = alt.Scale(
    domain=["total", "ganho", "perda"],
    range=[TINTA["apagada"], DIVERGENTE["pt"], DIVERGENTE["adversario"]],
)
PISO = 50.0  # o eixo começa em 50 milhões para os componentes ficarem visíveis
cascata = cascata.with_columns(
    pl.col("inicio").clip(lower_bound=PISO),
    rotulo=pl.when(pl.col("sinal") == "total")
    .then(pl.col("valor").round(1).cast(pl.String) + " mi")
    .otherwise(
        pl.when(pl.col("valor") > 0).then(pl.lit("+")).otherwise(pl.lit(""))
        + pl.col("valor").round(2).cast(pl.String)
    ),
    topo=pl.max_horizontal("inicio", "fim"),
)
barras = (
    alt.Chart(cascata)
    .mark_bar(cornerRadius=3, size=40)
    .encode(
        x=alt.X("passo:N", sort=ordem, title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
            "inicio:Q",
            title="Votos de Lula (milhões; eixo a partir de 50)",
            scale=alt.Scale(domain=[PISO, 59]),
        ),
        y2="fim:Q",
        color=alt.Color("sinal:N", scale=cores, legend=None),
        tooltip=[
            alt.Tooltip("passo:N", title="Passo"),
            alt.Tooltip("valor:Q", title="Milhões de votos", format="+.3f"),
        ],
    )
)
valores = (
    alt.Chart(cascata)
    .mark_text(dy=-8, fontSize=11, color=TINTA["secundaria"])
    .encode(x=alt.X("passo:N", sort=ordem), y="topo:Q", text="rotulo:N")
)
(barras + valores).properties(
    width=560, height=300, title="De 57,1 a 53,7 milhões: quase tudo é mudança de fatia"
)

**O que observar:** a fatia de Lula entre os válidos responde por −4,1 milhões; como o
eleitorado cresceu (+0,9 milhão), a perda líquida foi de 3,4 milhões. Comparecimento e
votos válidos quase não mexem. **A6 não se sustenta.**

**Limite (registrado):** isso é uma decomposição ecológica. Se, num mesmo município,
eleitores de Lula ficaram em casa e foram substituídos por novos eleitores de Flávio, isso
aparece aqui como "fatia". Separar as duas coisas exige dados individuais (painéis de
pesquisa).

### A6b · A abstenção cresceu mais nos redutos do PT?

Dentro da mesma região imediata, a variação do comparecimento (em logit) contra a força do
PT em 2022 (em logit). Coeficiente negativo = o comparecimento caiu mais onde o PT era forte.

In [ ]:
base = base.with_columns(
    t_de=pl.col("comparecimento_de") / pl.col("aptos_de"),
    t_para=pl.col("comparecimento_para") / pl.col("aptos_para"),
).with_columns(
    delta_logit_comparecimento=(pl.col("t_para") / (1 - pl.col("t_para"))).log()
    - (pl.col("t_de") / (1 - pl.col("t_de"))).log()
)
a6b = pf.feols(
    "delta_logit_comparecimento ~ pt_logit_de | cd_regiao_imediata",
    base.to_pandas(),
    vcov={"CRV1": "cd_regiao_imediata"},
)
a6b.tidy().loc[["pt_logit_de"], ["Estimate", "2.5%", "97.5%", "Pr(>|t|)"]]

**A6b se sustenta, mas o efeito é pequeno:** passar de um município onde o PT tinha 50% para
um onde tinha 73% (+1 em logit) reduz o comparecimento em ~0,25 pp. Não chega perto de
explicar a perda.

## A8 · Quanto da perda veio dos municípios pequenos?

**Critério registrado:** as AMCs com menos de 20 mil eleitores em 2022 respondem por uma
fração da perda maior que a sua fração do eleitorado.

In [ ]:
faixas = [5_000, 10_000, 20_000, 50_000, 100_000, 500_000]
rotulos = [
    "< 5 mil",
    "5–10 mil",
    "10–20 mil",
    "20–50 mil",
    "50–100 mil",
    "100–500 mil",
    "≥ 500 mil",
]
base = base.with_columns(
    faixa=pl.col("aptos_de").bin_intervals(faixas, labels=rotulos).cast(pl.String),
    pequena=pl.col("aptos_de") < 20_000,
)
a8 = (
    base.group_by("pequena")
    .agg(pl.col("delta").sum(), pl.col("aptos_de").sum(), pl.len().alias("AMCs"))
    .with_columns(
        (100 * pl.col("delta") / pl.col("delta").sum()).round(1).alias("% da perda"),
        (100 * pl.col("aptos_de") / pl.col("aptos_de").sum()).round(1).alias("% do eleitorado"),
    )
    .with_columns((pl.col("% da perda") / pl.col("% do eleitorado")).round(2).alias("razão"))
    .sort("pequena", descending=True)
)
a8

**Como ler:** para cada faixa de tamanho, a fração do eleitorado de 2022 (azul) e a fração
da perda de votos de Lula (laranja). Onde a barra laranja passa a azul, a faixa pesou na
perda mais do que o seu tamanho.

In [ ]:
por_faixa = (
    base.group_by("faixa")
    .agg(pl.col("delta").sum(), pl.col("aptos_de").sum(), pl.len().alias("amcs"))
    .with_columns(
        perda=100 * pl.col("delta") / pl.col("delta").sum(),
        eleitorado=100 * pl.col("aptos_de") / pl.col("aptos_de").sum(),
    )
    .unpivot(
        index=["faixa", "amcs"],
        on=["eleitorado", "perda"],
        variable_name="medida",
        value_name="pct",
    )
    .with_columns(
        pl.col("medida").replace_strict(
            {"eleitorado": "% do eleitorado de 2022", "perda": "% da perda de votos de Lula"}
        )
    )
)
medidas = ["% do eleitorado de 2022", "% da perda de votos de Lula"]
(
    alt.Chart(por_faixa)
    .mark_bar(cornerRadiusEnd=3, size=12)
    .encode(
        y=alt.Y("faixa:N", sort=rotulos, title="Eleitores aptos em 2022"),
        yOffset=alt.YOffset("medida:N", sort=medidas),
        x=alt.X("pct:Q", title="%"),
        color=alt.Color(
            "medida:N", sort=medidas, scale=alt.Scale(domain=medidas, range=SERIES[:2]), title=None
        ),
        tooltip=[
            alt.Tooltip("faixa:N", title="Faixa"),
            alt.Tooltip("medida:N", title="Medida"),
            alt.Tooltip("pct:Q", title="%", format=".1f"),
            alt.Tooltip("amcs:Q", title="AMCs"),
        ],
    )
    .properties(
        width=520,
        height=320,
        title="Os pequenos pesaram mais que o tamanho; os grandes, mais votos",
    )
)

### Por região e tamanho

**Como ler:** votos perdidos por Lula a cada 100 eleitores de 2022, em cada combinação de
região e faixa. Quanto mais escuro, maior a perda relativa. O tooltip mostra a perda em
votos.

In [ ]:
calor = (
    base.group_by("regiao", "faixa")
    .agg(pl.col("delta").sum(), pl.col("aptos_de").sum(), pl.len().alias("amcs"))
    .with_columns(
        por_100=-100 * pl.col("delta") / pl.col("aptos_de"), mil_votos=pl.col("delta") / 1e3
    )
)
ordem_regioes = ["N", "NE", "CO", "SE", "S"]
enc = alt.Chart(calor).encode(
    x=alt.X("faixa:N", sort=rotulos, title="Eleitores aptos em 2022", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("regiao:N", sort=ordem_regioes, title=None),
)
(
    enc.mark_rect(stroke="#fcfcfb", strokeWidth=2, cornerRadius=4).encode(
        color=alt.Color(
            "por_100:Q",
            scale=alt.Scale(range=["#cde2fb", DIVERGENTE["adversario"]]),
            title="Votos perdidos / 100 eleitores",
        ),
        tooltip=[
            alt.Tooltip("regiao:N", title="Região"),
            alt.Tooltip("faixa:N", title="Faixa"),
            alt.Tooltip("por_100:Q", title="Perdidos por 100 eleitores", format=".2f"),
            alt.Tooltip("mil_votos:Q", title="Variação (mil votos)", format="+,.0f"),
            alt.Tooltip("amcs:Q", title="AMCs"),
        ],
    )
    + enc.mark_text(fontSize=11).encode(
        text=alt.Text("por_100:Q", format=".1f"),
        color=alt.condition(
            "datum.por_100 > 3", alt.value("#ffffff"), alt.value(TINTA["primaria"])
        ),
    )
).properties(
    width=560, height=200, title="Votos de Lula perdidos a cada 100 eleitores, 2022 → 2026"
)

## Exploratório · E o adversário?

A A6 olha só para os votos de Lula. Mas a fatia de Lula entre os dois maiores também cai
quando o **adversário cresce**. Por região, a variação absoluta de votos de cada lado:

In [ ]:
regioes = (
    amc.join(municipios.group_by("amc_id").agg(pl.col("regiao").first()), on="amc_id")
    .group_by("regiao", "ano")
    .agg(pl.col("votos_pt", "votos_adv", "votos_outros", "aptos").sum())
    .pivot(on="ano", index="regiao", values=["votos_pt", "votos_adv", "votos_outros", "aptos"])
)
regioes.select(
    "regiao",
    *(
        ((pl.col(f"{c}_2026") - pl.col(f"{c}_2022")) / 1e6).round(2).alias(rotulo)
        for c, rotulo in [
            ("votos_pt", "Δ Lula (mi)"),
            ("votos_adv", "Δ adversário (mi)"),
            ("votos_outros", "Δ outros candidatos (mi)"),
            ("aptos", "Δ eleitorado (mi)"),
        ]
    ),
).sort(pl.col("regiao").replace_strict({r: i for i, r in enumerate(ordem_regioes)}))

**O que observar:** a queda do PT tem naturezas diferentes. No **Norte e no Nordeste, Lula
manteve os votos** (−0,3 milhão no Nordeste) e a fatia caiu porque o adversário cresceu
(+1,6 milhão no Nordeste), absorvendo eleitores novos e quem antes votava em outros. No
**Sudeste e no Sul, Lula perdeu votos de fato** (−1,8 e −1,0 milhão). Pista para uma hipótese
própria: o que o adversário conquistou no Nordeste foi eleitor novo ou eleitor de terceira via?

## Veredito

- **A6 não se sustenta.** Comparecimento e votos válidos respondem por ~6% da perda; o
  resto é mudança da fatia de Lula entre os válidos. O eleitorado cresceu e chegou a
  compensar parte da perda (+0,9 milhão).
- **A6b se sustenta com efeito pequeno:** o comparecimento caiu um pouco mais onde o PT era
  forte, sem relevância para o total.
- **A8 se sustenta:** as AMCs com menos de 20 mil eleitores têm ~22% do eleitorado e
  responderam por ~29% da perda (1,34× o seu peso). Mas **~71% dos votos perdidos vieram dos
  municípios maiores**: o gradiente por tamanho explica onde Lula perdeu *mais*, não onde
  perdeu *mais votos*.